# Module 2 Part 2: Bayesian Decision Theory

References:
* The ultimate reference (highly recommend): [Bayesian models of action and perception](https://www.cns.nyu.edu/malab/bayesianbook.html)
* Primer on Bayesian Decision Models & fitting them to data: [Ma '19
](https://www.cell.com/neuron/pdfExtended/S0896-6273(19)30840-2)
* Primer on the connections between Bayesian Decision Theory & RL: [Dayan & Daw '08](https://www.princeton.edu/~ndaw/dd08.pdf)
* Past tutorials that have inspired this one: [Hermundstad Cosyne '20](https://www.janelia.org/sites/default/files/Labs/Hermundstad%20Lab/Slides_Cosyne2020.pdf), [Ma Cosyne '19](http://www.cns.nyu.edu/malab/static/files/courses/Bayesian/Slides.pdf)


<br>

------


### This part of the tutorial has 3 (+2 bonus) sections

1. **Perceptual decisions**: Take what you have learnt about Bayesian perceptual inference in the previous module, and use it to make decisions about a stimulus
    1. In the face of unequal priors
    2. In the face of unequal costs/rewards - (using "utility")

2. **Hierarchical inference**: Add an extra level of Bayesian inference when there are more hidden variables of interest (e.g. the species or floral type that generated an observation)
    1. Inferring hidden species identity by observing spikes - introducing the concept of *marginalization*

3. **Inferring changing floral patches**: Use online Bayesian inference when the species generating observations changes over time

##### BONUS:

4. **Markov decision process (MDPs)**: Select actions when the state and nectar gradient are fully observable
5. **Partially observable Markov decision process (POMDPs)**: Select actions using beliefs when species identity is hidden

In [ ]:
# Import relevant modules and define some plotting functions
import pandas as pd
import numpy as np
import re
import seaborn as sns
from scipy.stats import norm, vonmises
import scipy.optimize as opt
from matplotlib.gridspec import GridSpec
from collections import OrderedDict
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import display, HTML
from typing import Dict, List

%matplotlib inline
sns.set_context('talk')

# Problem setup

**We are going to consider the decision processes of a virtual bee foraging in a flower patch containing two plant species, contexts ($c$): "species A" and "species B".** The two species tend to have different flower colours, but their colour distributions overlap, so a single flower is an ambiguous cue. Flowers can range in colour from red (0) to violet (1) or anywhere in between. We will refer to the colour of the flower as $f$. We will assume that flower colours $f$ are normally distributed within each species, but the two species differ in their average colour.


$$ p(f | c = \text{species A})  = \mathcal{N}(f; \mu_{\text{species A}}, \sigma^2)$$

$$ p(f | c = \text{species B})  = \mathcal{N}(f; \mu_{\text{species B}}, \sigma^2)$$


The bee needs to know how to maximise its reward. However, it does not directly observe species identity. It mainly observes noisy colour cues, and because the two species overlap in colour, it must infer which species it is sampling from. This matters because the species can differ in expected nectar, so identifying the likely species helps the bee choose where to forage.


To help us, let's set up a class with some methods that will let us easily create and sample an encountered flower colour $f$ from the two species:


In [ ]:
class Context:
    
    # initialize
    def __init__(self, pFlowerMu, pFlowerSig = 0.1):
        self.pFlowerMu = pFlowerMu   # mean of the flower color distribution
        self.pFlowerSig = pFlowerSig  # standard deviation of flower color distribution

    # method 1: Probability density of flower colors 
    def pFlower(self, f):
        return norm.pdf(
            f, 
            loc = self.pFlowerMu, 
            scale = self.pFlowerSig)

    # method 2: Cumulative density of flower colors 
    def cFlower(self, f):
        return norm.cdf(
            f,
            loc = self.pFlowerMu, 
            scale = self.pFlowerSig)

    # method 3: Randomly sample a flower color 
    def sample(self, n = 1):
        return norm.rvs(
            loc = self.pFlowerMu, 
            scale = self.pFlowerSig,
            size = n)


Now with this class, let's create the two species, $c \in \{\text{species A},\text{species B}\}$, which differ in the average colour of flowers $f$:


In [ ]:
# make a dictionary for easily referring to the two species
contexts = {
    'species_A': Context(pFlowerMu = 0.66), 
    'species_B': Context(pFlowerMu = 0.33) 
    }

# define some useful variables for plotting
colors = {
    'species_A':[0,0,1], 
    'species_B':[1,0,0]
    }

# possible values of f
f = np.arange(0, 1, 0.01)

# And then plot
fig = plt.figure(figsize=(8, 3))
axs = fig.gca()
for c in contexts:
    axs.fill(
        f,
        contexts[c].pFlower(f),
        alpha = 0.3,
        color = colors[c],
        label = c)
axs.legend()
axs.set_xlabel('Flower color')
axs.set_ylabel('Probability density')
axs.set_yticks([])
axs.set_title('Distribution of flower colors')
sns.despine()
plt.show()


# 1. Perceptual decisions - which species did this flower likely come from?

To start, the first piece of information the bee might need is the likely species of the flower it has encountered. Given an observed flower colour, the bee can estimate the probability that the flower came from species A or species B.

In other words, given a flower colour cue, we want to classify the flower into the species that most likely generated it. This is more akin to some decision-making tasks we might set up in the lab.


So first, using what you've learnt about Bayesian inference, write a function that returns the likelihood function - that is the probability density of any given flower colour (hence a function of flower colour) given that the flower came from species A or species B.

$$ L(f | c = \text{species A}) = p(f | c = \text{species A}) $$

Here's a schematic of this generative model, with solid bubbles being observables and empty ones being hidden.

<div style="text-align:center">
<img src="./model1.png" width="100"/>
</div>


In [ ]:
def lik_function(f, context_name:str, contexts: Dict[str, Context]):
    """
    Return the likelihood density of an observed flower color given a species.
        f: observed flower color(s)
        context_name: name of the candidate species
        contexts: species and their flower-color distributions
    Returns:
        likelihood: likelihood density of the observed flower colors
    """
    
    # write the expression for computing the likelihood function here

    # FILL ME IN!
    
    return likelihood


Using this function, we're going to write down a *decision rule* (i.e. a way of deciding which species generated the observed flower colour) that *maximizes* this function. We can express this operation of maximising likelihood in a few different equivalent ways:
* Picking the species $c$ that has a higher likelihood $L(f | c)$:
$$\text{choose species A if } L(f | c = \text{species A}) > L(f | c = \text{species B})$$
* Picking $\text{species A}$ if the *likelihood ratio* (LR) of species A to species B is >1:
$$\text{choose species A if LR} \equiv \frac{L(f | c = \text{species A})}{L(f | c = \text{species B})}>1$$
* Taking the log, and picking $\text{species A}$ if the *log likelihood ratio* (LLR) is >0:
$$\text{choose species A if LLR}  \equiv log \bigg( \frac{L(f | c = \text{species A})}{L(f | c = \text{species B}} \bigg)>0$$


Let's plot the likelihood function you wrote, and also examine the log likelihood ratios of 1000 observed flower colours, 500 from each species!



In [ ]:
# Convenient method to plot any given function of flower color for both species
def plot_func_of_f_c(func, func_name: str, f, contexts, colors, **func_kwargs):
    fig = plt.figure(figsize=(8, 4))
    y = dict()
    for c in contexts:
        y[c]=func(f,c, contexts, **func_kwargs)
        plt.plot(f, y[c], '-', color = colors[c], label = c)
    threshold = min(f[y['species_A']>y['species_B']])
    plt.vlines(threshold,min([min(y['species_A']), min(y['species_B'])]),max([max(y['species_A']), max(y['species_B'])]),linestyles='dashed',colors='k')
    
    plt.xlabel('Flower color')
    plt.ylabel(f"{func_name}")
    plt.legend()
    plt.title(f"{func_name} as a function of flower color and species")
    sns.despine()
    return threshold

threshold = plot_func_of_f_c(lik_function, 'Likelihood', f, contexts, colors)


How do you think we can implement the decision rule of *maximizing* this function? To build intuition, let's simulate some datapoints and examine the log likelihood ratio (LLR)

In [ ]:
nsamples = 1000

loglik_ratios = dict()
data = dict()
for c in contexts:
    data[c] = contexts[c].sample(n = int(nsamples/2))
    loglik_ratios[c] = # FILL ME IN!
    plt.scatter(data[c], loglik_ratios[c], color = colors[c],alpha=0.1)
plt.xlim([0,1])
plt.xlabel('Flower color')
plt.ylabel('Log likelihood ratio')
plt.title('Log likelihood ratios of species A v.s. species B')
plt.hlines(0,0,1,colors='k')
plt.vlines(threshold, min(loglik_ratios['species_B']), max(loglik_ratios['species_A']), linestyles='dashed',colors='k')
sns.despine()


Notice that the log likelihood ratio (LLR) is *linear* with respect to the observations! It did not have to be this way! Here, this happens because the two likelihoods are Gaussian and have the same variance.

This means that, to implement our decision rule of thresholding the LLR at 0, we can equivalently threshold the observed flower color (in this case at approximately $f=0.5$).


### Incorporating prior information

If we use Bayesian reasoning, then there is more to the model than the likelihood: we must consider our prior probability that a sampled flower belongs to each species.  One species might be more common than another in the bee's environment, so we should take this information into consideration.

Let's write the *posterior* probability that a flower came from species A versus species B given the observations, by combining the likelihood and prior.

HINT: Remember that the posterior probability is a *probability* so it must be normalized!

<br>

$$ \begin{align}
p(c  = \text{species A}| f) &∝ p(f | c= \text{species A}) \times p(c= \text{species A}) \\
 &= \frac{p(f | c= \text{species A}) \times p(c= \text{species A})}{p(f | c= \text{species A}) \times p(c= \text{species A}) + p(f | c= \text{species B}) \times p(c= \text{species B})}
 \end{align}$$




In [ ]:
def posterior_probability(f, chosen_context:str, contexts: Dict[str,Context], priors:Dict[str,float]):
    """
    Return the normalized posterior probability of a species given an observed flower color and priors.
        f: observed flower color(s)
        chosen_context: species for which posterior is being evaluated
        contexts: dictionary of Context classes
        prior: prior probability of that species
    Returns:
        posterior: posterior probability of the chosen species
    """
    # write the expression for computing the posterior here

    # FILL ME IN!
    
    return posterior_chosen/denom

Let's look at the posterior function when the prior probabilities of the two species are unequal. What do you expect to see? Think through it before you plot.


In [ ]:
priors = {
    'species_A': 0.9,
    'species_B': 0.1
}
threshold = plot_func_of_f_c(posterior_probability, 'Posterior probability', f, contexts, colors, priors = priors)
plt.vlines(0.5,0,1,linestyles='dotted')
plt.show()

What should happen at "ambiguous" flower colours of 0.5 when the prior is skewed? Should this affect the decision rule, and if so how?

> If species A is more common in this patch, then ambiguous flower colours should bias the bee toward species A.
>
> If we express our decision rule as a threshold on the log *posterior* ratio, that is we choose species A when the log posterior ratio is >0, and choose species B when the log posterior ratio is <0 - then this decision-rule remains the same when the prior changes and still yields *optimal* decisions!
>
> However if we instead express our decision rule as a threshold on the sensory observations, it gets *biased* when the prior is skewed.


## Utilities

Ok, now we are going to move even closer to real-life decisions.

For each possible true species, there are two possible classifications the bee could make, one correct and one incorrect. However, these may not be equally good/bad - different errors could have different costs, similarly different correct decisions could have different rewards. For example, mistaking a low-nectar species for a high-nectar species may waste time, while correctly identifying a high-nectar species may be especially valuable.

Here, `rewards[action]` is the reward when the chosen species is correct, while `costs[action]` is the loss when that chosen species is incorrect.

Bayesian decision theory tells us to choose the action with the greater expected utility:

$$
U(\text{choose A} \mid f) = r_A P(A \mid f) - c_A P(B \mid f)
$$

$$
U(\text{choose B} \mid f) = r_B P(B \mid f) - c_B P(A \mid f)
$$

Choose species A when $U(\text{choose A} \mid f) > U(\text{choose B} \mid f)$.

This introduces additional variables: the action we take $a$ (choose species A, choose species B), and the reward $r$ / cost $c$ of correct/incorrect decisions.

<div style="text-align:center">
<img src="./model2.png" width="100"/>
</div>


Write this function below and play with different costs and rewards.


In [ ]:
def utility_function(f, chosen_context:str, contexts: Dict[str, Context], priors:Dict[str,float], rewards:Dict[str,float], costs:Dict[str,float]):
    """
    Return the expected utility of choosing a species given a flower color.
        f: observed flower color
        contexts: species classes
        priors: prior probabilities for each species
        rewards: reward for each action when it is correct
        costs: loss for each action when it is incorrect
    Returns:
        utility: utility function for choosing that context
    """
    # Compute posteriors for both contexts
    unchosen_context = [c for c in contexts if c!=chosen_context][0]
    posterior_chosen = posterior_probability(f,chosen_context, contexts, priors)
    posterior_unchosen = posterior_probability(f,unchosen_context, contexts, priors)
    
    # Compute utility for chosen context
    utility = # FILL ME IN!
    
    return utility


Let's look at the expected utilities for a number of observed flower color samples, given prior probabilities, rewards, and associated costs. Play around with these values to see what happens:


In [ ]:
# set the prior to be high for species_A and 
# the cost of incorrectly choosing species_A when the species is really species_B to be really high

priors = {
    'species_A': 0.9,
    'species_B': 0.1
}

rewards = { # of correctly choosing each species
    'species_A': 1,
    'species_B': 1
}

costs = { # of incorrectly choosing each species
    'species_A': 91,
    'species_B': 11
}


threshold = plot_func_of_f_c(utility_function, 'Utility function', f, contexts, colors, priors = priors, rewards=rewards, costs=costs)
plt.vlines(0.5,-90,0,linestyles='dotted')
plt.show()


#### What do you notice about the interplay between prior ratios and reward ratios?

> They can compensate for each other! This means two things:
>
> 1. *Optimal* decisions can be biased towards one category for a few different reasons: 
>       - that category is more prevalent i.e. a higher prior
>       - the reward for correctly choosing it is high 
>       - the cost of incorrectly choosing the alternative category is high.
> >
> 2. Consequently, it is difficult to identify why an agent made a given decision, even if we assume the agent is choosing optimally.
>

## Takeaways

Bayesian Decision Theory (BDT) extends Bayesian inference to *decisions* i.e. discrete choices between (unobservable) categories, based on observed evidence. 

In BDT, we consider not just the (posterior) probabilities of these categories given the evidence, but also the rewards/costs of making correct/incorrect decisions. 
These two ingredients combine to form the *utility* of different decisions, and BDT lets us make decisions that maximize this utility.

Consequently, BDT is a reminder of the fact that optimal decision-making agents can be biased towards one decision or another for many different reasons - because that decision is more prevalent, or because the associated rewards/costs are high.

# 2. Hierarchical inference

So far, we've assumed direct access to the flower colours $f$. However, in reality we may only have access to noisy sensory observations $x$ from a colour-sensitive neuron. This modifies our generative model to a new hierarchical graph:

<div style="text-align:center">
    <img src="./model3.png" alt="Generative Model" width="100"/>
</div>



Let's assume that the color detector neuron has firing rate centered around the true color, with sensory noise characterized by a variance of $\sigma_{\text{sensory}}^2$ The tuning function of such a neuron can be written as:

In [ ]:
def tuningFun(f, beePars):
    # Gaussian noise on firing rate centered at the color
    rate = f + np.random.normal(0, beePars['sigmaSensory'], size=f.shape) 
    return rate

In this case, we want to *marginalize* over the true flower colour, since we still want to infer the variable of interest $c$: the hidden species. That is, we are interested in

$$
P(x \mid c) = \int P(x \mid f) P(f \mid c) \, df.
$$

If we assume that both $P(x \mid f)$ and $P(f \mid c)$ are Gaussian with variances $\sigma_{\text{sensory}}^2$ and $\sigma_{f}^2$ respectively, 
$$P(x \mid f) = \mathcal{N}(x | f, \sigma^2_\text{sensory})$$
$$P(f \mid c) = \mathcal{N}(f \mid \mu_c, \sigma^2_\text{f})$$

then the integral also becomes Gaussian - and assuming that the firing rate distribution is centered around the true flower colour, this new Gaussian has variance $\sigma_{f}^2 + \sigma_{sensory}^2$. (This is an important and very useful property of the Gaussian distribution!)

Before we go on and code the marginal distribution, let's define the parameters of the two species and the bee.


In [ ]:
### Parameters of the bee's *internal* distribution of noisy sensory observations
beePars = {'sigmaSensory': 0.1}

### In contrast, these are the parameters of the *environmental* distribution of flower colors
contexts = {
    'species_A': Context(pFlowerMu = 0.66, pFlowerSig = 0.1), 
    'species_B': Context(pFlowerMu = 0.33, pFlowerSig = 0.1) 
    }

# define some useful variables for plotting
colors = {
    'species_A':[0,0,1], 
    'species_B':[1,0,0]
    }

Complete the function below to compute marginal likelihood $P(x|c)$, where $c$ is the hidden species:


In [ ]:
def marginal_lik(x, context, beePars):
    
    # FILL ME IN!
    
    return marginal_lik


#### Monte Carlo techniques to approximate marginalization

In this specific case, because each of our sources of noise was Gaussian and independent, we could derive the integral to get to the marginal distribution. However, in most real-world cases, this integral becomes intractable to evaluate analytically.  If this happens, we can use *simulation* to approximate it by sampling from the distribution. Such approaches are called Monte Carlo techniques (due to the use of randomness, it was named after a famous casino in Monaco) and have wide applications in real world problems.

$$
P(x \mid c) \approx \frac{1}{N} \sum_{i=1}^{N} P(x \mid f_i), \quad \text{where} f_i \sim P(f \mid c)
$$

In [ ]:
def marginal_lik_sim(x, context, beePars):
    f_sample = context.sample(n = 100)
    sigma_sensory = beePars['sigmaSensory']
    p_sample = # FILL ME IN!
    return p_sample.mean(axis = 0)


Let us plot both the analytical and simulation-based marginal distributions to confirm that they match:


In [ ]:
def compare_analytical_simulated_marginal_likelihoods(contexts, beePars):
    
    fig, axs = plt.subplots(1, 2, figsize=(14, 7))

    for c in contexts:

        # Plot observed firing rates for different true flower colors
        data = contexts[c].sample(n=100)
        rate = tuningFun(data,beePars)
        axs[0].plot(data, data, color = 'k')
        axs[0].set_ylim([0,1])
        axs[0].scatter(data, rate, color = colors[c], alpha = 0.3)
        axs[0].set_xlabel('True flower color')
        axs[0].set_ylabel('Firing rate $(x)$')
        axs[0].set_title('Firing rates $(x)$ observed\nfor the two species')

        # We will compute analytical and simulated marginal distributions
        # over all these possible values of x (firing rate)
        x = np.linspace(rate.min(),rate.max(),100)

        # Plot analytical marginal distribution
        axs[1].plot(
            marginal_lik(x, contexts[c], beePars),
            x, 
            color = colors[c],
            alpha= 0.3, 
            label='analytical')
        
        # Plot simulated marginal distribution
        axs[1].plot(
            marginal_lik_sim(x, contexts[c], beePars),
            x, 
            ls = '--',
            color = colors[c],
            alpha= 0.3, 
            label = 'monte carlo')

        # Plot the distribution of firing rates for the two species
        label = f"$P(x | c = {c})$"
        axs[1].hist(
            rate,
            color = colors[c],
            alpha= 0.3,
            label = label, 
            orientation='horizontal', 
            density=True)
        axs[1].set_xlim([0,5])
        axs[1].set_xticks([])
        axs[1].legend()
        axs[1].set_ylabel('Firing rate')
        axs[1].set_xlabel('marginal likelihood $P(x|c)$')
        axs[1].set_title('Comparing analytical vs simulated\nmarginal likelihood $P(x|c)$') 

    sns.despine()
    plt.tight_layout()
    plt.show()
    
compare_analytical_simulated_marginal_likelihoods(contexts, beePars)

So, here given a noisy firing rate, we can infer the hidden species using the marginal likelihood, marginalizing over all possible flower colours. For high firing rates (or intense flower colours), the marginal likelihood is larger for species A - where the true mean was 0.66, and vice versa. To make a *decision* about the species, just like before it will be helpful to compute the ratio of the two marginal likelihoods:


In [ ]:
def marginal_lik_ratio(x, contexts, beePars):
    marginal_lik_A = marginal_lik(x, contexts['species_A'],beePars)
    marginal_lik_B = marginal_lik(x,contexts['species_B'],beePars)
    result = # FILL_ME IN!
    return result

Play around with the sensory noise $\sigma_{\text{sensory}}$ to see the effect on (1) marginal likelihoods and then (2) marginal likelihood ratios!

(1) marginal likelihoods

In [ ]:
beePars = {'sigmaSensory': .5} # try values: [0.1,0.2,0.4,0.8]

compare_analytical_simulated_marginal_likelihoods(contexts, beePars)

(2) marginal likelihood ratios

In [ ]:
# possible firing rates of color detector neuron.
x = np.linspace(0,1,100)

plt.figure(figsize=(5, 5))
legend_label = "$\\sigma_{sensory}$"

# vary the sensory noise of the bee
for sigmaSensory in [0.1,0.2,0.4,0.8,0.9]:
    
    beePars = {'sigmaSensory': sigmaSensory}
    plt.plot(
        x, 
        np.log(marginal_lik_ratio(x, contexts, beePars)), 
        label = f"{legend_label}={sigmaSensory}"
        )
plt.xlabel('Firing rate $(x)$')
plt.ylabel('Marginal log likelihood ratio')
plt.title("Effect of sensory noise on marginal LLR")
sns.despine()
plt.legend(bbox_to_anchor=[1.1, 0.8])
plt.show()

### Think about
- What happens to marginal likelihoods and marginal likelihood ratios? 
- Where should the decision boundary be drawn? 
- Is the bee still able to *correctly* decide which species it might be sampling from?


# 3. Inferring changing floral patches
Ok, now we will let the bee fly around. Nearby flowers tend to come from the same species, but the dominant species can change as the bee moves through the patch. We will model the decision dynamics of the bee in such changing local floral patches.

<div style="text-align:center">
    <img src="./model4.png" alt="Generative Model" width="400"/>
</div>

**BUT** before going there, we will learn how to do online Bayesian inference (also called recursive Bayes or Bayesian filtering) in a more restricted setting - one in which the local dominant species changes without the bee having any control over which flowers it samples. This is like the bee flying in a straight line and sampling from each flower it sees: it passes through regions with each of the two dominant species, but it does not make any choice about where to sample.


#### First, we will define a class that lets us simulate this situation
In addition to the parameters we have had for the two species (namely $\mu_c$) and bee parameters (sensory noise $\sigma_{\text{sensory}}^2$), we are going to introduce another parameter $p_{switch}$ which is the probability that the local dominant species changes as the bee flies in a line. We assume this process is Markovian: there is a fixed probability of change occurring at every time step.

<div style="text-align:center">
    <img src="./model5.png" alt="Generative Model" width="400"/>
</div>



In [ ]:
class simulateEnv:
    
    def __init__(self, contexts, beePars, **envParams):
        self.contexts = contexts
        self.beePars = beePars
        self.switchType = envParams.get('switchType', None)
        self.nSwitch = envParams.get('nSwitch', None)
        self.contextLabels = list(self.contexts)


    #Sample contexts, observed flower colours aka firing rates
    def sample(self, tMax):
        
        fObs = np.zeros(tMax)  # observations made by the bee i.e. firing rates of the color neuron
        pSwitch = 1/self.nSwitch # Hazard rate i.e. 1/(avg no. encounters before switching contexts)
        contextFlag = np.zeros(tMax, dtype='int') # store which context the bee is in (environmental dynamics)

        # Simulate observations of flower colors for this assumed rate of context change
        
        # sample the initial context 
        contextFlag[0] = np.random.randint(len(self.contextLabels))
        context_0 = self.contextLabels[contextFlag[0]]
        
        # make a firing rate observation in this context (remember marginal likelihood is P(x|c))
        fObs[0] = self.sample_from_marginal_lik(context_0, 1)[0]

        switch_indicator = False
    
        # simulate over time
        for i in range(1, tMax):
            
            # switch every nSwitch trials
            if self.switchType == 'periodic':
                switch_indicator = np.mod(i, self.nSwitch) == 0
                
            # Switch contexts with a probability of 'pSwitch'
            elif self.switchType == 'markov':
                switch_indicator = np.random.rand() < pSwitch
            else:
                raise ValueError(
                    "unknown switchType; expected 'periodic' or 'markov'"
                )
            
            # switch contexts
            if switch_indicator:
                contextFlag[i] = not(contextFlag[i-1])  
            else:
                contextFlag[i] = contextFlag[i-1]
                
            # Sample flower colors from chosen context
            fObs[i] = self.sample_from_marginal_lik(self.contextLabels[contextFlag[i]], 1)[0]
            
            
        return contextFlag, fObs
    
    
    # method to sample observations (firing rates) given context
    def sample_from_marginal_lik(self, context, n):
        
        sample = norm.rvs(
            loc = self.contexts[context].pFlowerMu,
            scale = np.sqrt(self.contexts[context].pFlowerSig**2 + self.beePars['sigmaSensory']**2),
            size = n
        )
    
        return sample    
    
    
    

# plotting function for visualizing firing rates when the contexts are changing
def plot_context_and_observations(tMax, contextFlag, fObs, title):

    # Plot true context and sensory observations with two y-axes
    fig, ax1 = plt.subplots(figsize=(12, 3))

    # First subplot (true species)
    scatter1 = ax1.scatter(range(tMax), contextFlag, 5, color='k', label='True species')
    ax1.set_yticks([0, 1])
    ax1.set_yticklabels(list(contexts))
    ax1.set_xlabel('Time')

    # Second y-axis for the second subplot (Sensory observations)
    ax2 = ax1.twinx()
    this_color = [0.5, 0.5, 0]
    scatter2 = ax2.scatter(range(tMax), fObs, 8, color=this_color, label='Firing rate observations')
    ax2.set_ylabel('Firing rate')
    ax2.spines['right'].set_color(this_color)
    ax2.tick_params(axis='y', colors = this_color)
    ax2.yaxis.label.set_color(this_color)

    lines = [scatter1, scatter2]
    labels = [line.get_label() for line in lines]
    ax1.legend(
        lines, 
        labels, 
        bbox_to_anchor = (0.8, -0.4),
        ncol = 2
        )
    
    ax1.set_title(f"{title} environment")
    sns.despine(right = False)
    
    return fig, ax1



Okay, let's visualize this change: you can change the `switchType` to `periodic` in the cell below, to very clearly visualize distributional shifts in observations due to changing local species dominance. Also try increasing the sensory noise to see how that changes the observations that the bee makes.


In [ ]:
## Create a bee with a chosen sensory noise parameter - PLAY WITH THIS!
beePars = {'sigmaSensory':0.1}

### Parameters of the *environmental* distribution of flower colors
contexts = OrderedDict({
    'species_A': Context(pFlowerMu = 0.66, pFlowerSig = 0.1), 
    'species_B': Context(pFlowerMu = 0.33, pFlowerSig = 0.1) 
    })

### number of time points
tMax = 200

### Simulate species switches, with an average change every 20 trials
# Change switchType to 'periodic' to easily understand the relationship between species switch and firing rates
envPars = {
    'switchType': 'markov',
    'nSwitch': 20 
    }

# initialize class
sim = simulateEnv(contexts, beePars, **envPars)

### Sample species, firing rate samples from that species
contextFlag,fObs = sim.sample(tMax)
fig, ax = plot_context_and_observations(tMax, contextFlag, fObs, envPars['switchType'])


### Inference 

Now let's write a function to perform online inference as the bee encounters flowers in different local floral patches.

Some terminology first:
The posterior probability that the bee thinks the current local patch is species B, given the sequence of flowers it has encountered up to and including time t, is often referred to as the belief: $p_{\text{belief}, t}^{\text{species B}}$.

This belief will need to be updated at each time step, based on the dynamics of species switches and the encountered flower colours.

Now, when the bee encounters its first flower, two steps need to occur:
1. Prediction: probability that the current flower is still likely to come from species B, given that the bee encountered a flower nearby (irrespective of its colour)

2. Update: probability that the current flower is still likely to come from species B, given that the bee observed the specific colour of the flower.

Let's write down what these steps look like in math:

##### **the prediction step**:
$$p_{\text{predict}, t = 1}^{\text{species B}} = p^{\text{species B}}_{\text{belief},t=0}(1-p_{\text{switch}}) + (1 - p^{\text{species B}}_{\text{belief},t=0})p_{\text{switch}} $$
This first term here evaluates the probability of sampling species B if a switch didn't happen. And the second term computes the probability of sampling species B if a switch happened.

##### **the update step**:
our prediction acts as the prior and we compute the likelihood of observing firing rate $x_1$ given we are sampling species B to arrive at the posterior probability of thinking that the current flower came from species B
$$p_{\text{belief}, t = 1}^{\text{species B}} \propto p_{\text{predict}, t = 1}^{\text{species B}} \mathcal{N}\left(x_1; \mu_{\text{species B}}, \sigma^2_{\text{species B}} + \sigma^2_{\text{sensory}}\right) $$

Again, this process will be repeated at every time step. Therefore, essentially in Bayesian filtering **"today’s posterior acts as tomorrow’s prior”** (Lindley, Bayesian statistics, a review. 1972, p. 2). In this example, the posterior after one flower becomes the prior expectation for the next nearby flower. This is comparable to evidence accumulation frameworks, such as the DDM, where the observer is thought to accumulate noisy samples of evidence over time. Here, the evidence is the log-likelihood, and accumulation is the Bayes-optimal solution.




In [ ]:
# Perform online inference
def onlineInference(fObs, contexts, beePars, pSwitch=0.5, p0=(0.5, 0.5)):

    # Construct transition matrix
    T = np.array([
        [1-pSwitch, pSwitch],
        [pSwitch, 1-pSwitch],
    ])

    # Initialize beliefs
    pContext = np.zeros([2, np.size(fObs)])
    contextLabel = list(contexts)

    def observation_likelihood(observation):
        return np.array([
            marginal_lik(observation, contexts[label], beePars)
            for label in contextLabel
        ])

    # Start with the original prior. For each observation, update the
    # current belief and store the resulting posterior.
    belief = np.asarray(p0, dtype=float)

    for t in np.arange(np.size(fObs)):
        # Before observations after the first, predict whether the species
        # remained the same or switched.
        if t > 0:
            belief = np.matmul(T, belief)

        belief = # FILL ME IN!
        belief = belief / belief.sum() # Normalise
        pContext[:, t] = belief

    return pContext





Now, let's plot the running posterior belief as well as the posterior belief averaged over switches.

In [ ]:
# Create a bee with a chosen sensory noise parameter - PLAY WITH THIS!
beePars = {'sigmaSensory': 0.1}

### Parameters of the *environmental* distribution of flower colors
contexts = OrderedDict({
    'species_A': Context(pFlowerMu = 0.66, pFlowerSig = 0.1), 
    'species_B': Context(pFlowerMu = 0.33, pFlowerSig = 0.1) 
    })

# initialize class
sim = simulateEnv(contexts, beePars, **envPars)

# Sample species, firing rate samples from that species
contextFlag,fObs = sim.sample(tMax)

# infer posterior beliefs
pContext = onlineInference(
    fObs, 
    contexts, 
    beePars, 
    pSwitch = 1/envPars['nSwitch'],
    # pSwitch = 0.5,
    p0 = [0.5, 0.5]  # initial beliefs about the two species
    )

# plot the beliefs
fig, axs = plot_context_and_observations(tMax, contextFlag, fObs, envPars['switchType'])
species_b_index = list(contexts).index('species_B')
label = r'$p(\mathrm{species\ B})$'
axs.plot(pContext[species_b_index, :], color='r', label=label)
axs.legend( bbox_to_anchor = (1.4, 0.7),)
plt.show()

# Plot posterior belief averaged over switches
pattern = {'Species A->Species B':'0000011111', 'Species B->Species A':'1111100000'}
fig2 = plt.figure(figsize=(16.5, 3))
k = 1
for s in pattern.keys():
    fig2.add_subplot(1, 2 ,k, title = 'Average p('+s+')')
    k+=1
    pAvg = np.zeros(10)
    inds = [m.start() for m in re.finditer(pattern[s], ''.join(str(e) for e in contextFlag))]
    for i in inds:
        pAvg = pAvg + pContext[species_b_index, i:i+10]
    pAvg = pAvg/np.size(inds)
    plt.plot(np.arange(-4,6), pAvg, c='r')
    plt.ylabel('p (species_B)')
    plt.plot([0,0],[0,1],'k--')
sns.despine()


#### things to think about:
1. What happens if you increase the sensory noise, observation noise?
2. What happens if the bee thinks $p_{\text{switch}}$ is higher than it is?
3. What happens if the bee's knowledge of the distribution of flower colours in the two species is not accurate? 


# 4. Selecting actions in a Markov decision process
Let's turn now to actions! So far, the bee has encountered flowers without controlling which flower it samples next. We will now let its actions change its location in the patch.  For this, we will assume that we have a 2-dimensional patch that the bee can navigate around.

We will start with the simpler case: there is only one flower species, flower colour varies smoothly over space, and nectar increases with flower colour. Throughout this MDP example, every flower belongs to that same species. The bee directly observes its location, heading, and the colour field. Its task is to move through the patch to the flower with the most nectar.

A **Markov decision process** (MDP) consists of states, actions, transitions, and rewards. Here:

- the state is the bee's position and heading;
- an action changes its heading and position;
- the transition specifies the next position and heading;
- the reward is the nectar at the next flower.

<div style="text-align:center">
    <img src="./model7.png" alt="Markov decision process" width="900"/>
</div>

The next state and reward depend only on the current state and action, and the complete state is observable. This makes the environment a proper MDP.

In [ ]:
# Relative turns available to the bee. Heading 2 points to the right.
headingVectors = np.array([
    [-1, 0], [-1, 1], [0, 1], [1, 1],
    [1, 0], [1, -1], [0, -1], [-1, -1]
])
turnSteps = np.array([-2, -1, 0, 1, 2])
turnLabels = ['-90', '-45', '0', '+45', '+90']


def makeSpatialField(X, Y):
    rawField = (
        1.00 * np.exp(
            -0.55 * (X - 1.45)**2
            -0.70 * (Y - 1.25)**2
            +0.18 * (X - 1.45) * (Y - 1.25)
        )
        +0.82 * np.exp(
            -0.75 * (X + 1.75)**2
            -0.55 * (Y - 1.85)**2
            -0.15 * (X + 1.75) * (Y - 1.85)
        )
        +0.68 * np.exp(
            -0.65 * (X + 1.55)**2
            -0.85 * (Y + 1.65)**2
            +0.12 * (X + 1.55) * (Y + 1.65)
        )
        +0.58 * np.exp(
            -0.95 * (X - 2.05)**2
            -0.65 * (Y + 1.75)**2
        )
        -0.22 * np.exp(-0.45 * X**2 - 0.55 * Y**2)
    )
    return (
        (rawField - rawField.min())
        / (rawField.max() - rawField.min())
    )


class MDPFlowerPatch:
    def __init__(self, gridSize=21):
        coordinates = np.linspace(-3, 3, gridSize)
        self.X, self.Y = np.meshgrid(coordinates, coordinates)

        # One species with a smooth flower-colour field and local peaks.
        self.F = makeSpatialField(self.X, self.Y)
        self.nectarField = self.nectar(self.F)
        self.maximumIndex = np.unravel_index(
            np.argmax(self.nectarField), self.nectarField.shape
        )

    def nectar(self, flowerColor):
        return norm.cdf(flowerColor, loc=0.5, scale=0.15)

    def nextState(self, state, action):
        row, column, heading = state
        if (row, column) == self.maximumIndex:
            return state

        headingNext = int(
            (heading + turnSteps[action]) % len(headingVectors)
        )
        rowStep, columnStep = headingVectors[headingNext]
        rowNext = int(np.clip(row + rowStep, 0, self.F.shape[0] - 1))
        columnNext = int(
            np.clip(column + columnStep, 0, self.F.shape[1] - 1)
        )
        return rowNext, columnNext, headingNext

    def nectarAt(self, state):
        return float(self.nectarField[state[0], state[1]])

Let's take a look at this patch. Because there is only one species, colour is not evidence about a hidden context. It directly specifies the nectar available at each location. The colour and nectar fields therefore have the same global maximum.

In [ ]:
mdpPatch = MDPFlowerPatch(gridSize=21)
extent = [
    mdpPatch.X.min(), mdpPatch.X.max(),
    mdpPatch.Y.min(), mdpPatch.Y.max()
]

fig, axs = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
fieldImage = axs[0].imshow(
    mdpPatch.F,
    origin='lower',
    extent=extent,
    interpolation='bilinear',
    cmap='coolwarm_r',
    vmin=0,
    vmax=1
)
fig.colorbar(fieldImage, ax=axs[0], label='Flower colour')
maximumRow, maximumColumn = mdpPatch.maximumIndex

axs[0].set_title('One-species flower-colour field')
axs[0].set_xlabel('x position')
axs[0].set_ylabel('y position')

fPossible = np.linspace(0, 1, 200)
axs[1].plot(fPossible, mdpPatch.nectar(fPossible), color='black')
axs[1].set_title('Nectar function')
axs[1].set_xlabel('Flower colour')
axs[1].set_ylabel('Nectar')
sns.despine(ax=axs[1])
plt.show()

We will therefore use a simple one-step policy. For each possible turn, the bee evaluates the nectar at the resulting location and passes these values through a softmax.

In [ ]:
class BeeMDP:
    def __init__(self, params):
        self.params = params

    def actionValues(self, state, patch):
        return np.array([
            patch.nectarAt(patch.nextState(state, action))
            for action in range(len(turnSteps))
        ])

    def actionProbabilities(self, actionValues):
        logits = self.params['softmaxInverseTemp'] * actionValues
        probabilities = np.exp(logits - np.max(logits))
        return probabilities / probabilities.sum()

    def policyFunc(self, actionValues, rng):
        probabilities = self.actionProbabilities(actionValues)
        action = int(rng.choice(len(actionValues), p=probabilities))
        return action, probabilities


class SimulateBeeMDP:
    def __init__(self, bee, patch, simPars):
        self.bee = bee
        self.patch = patch
        self.simPars = simPars
        self.rng = np.random.default_rng(simPars['seed'])
        self.state = simPars['stateInit']
        self.trajectory = [self.state]
        self.actionProbabilities = np.ones(len(turnSteps)) / len(turnSteps)
        self.done = self.state[:2] == self.patch.maximumIndex
        self.fig, (self.axp, self.axv) = plt.subplots(
            1, 2, figsize=(12, 5), layout='constrained'
        )

    def update(self, t):
        if not self.done:
            actionValues = self.bee.actionValues(self.state, self.patch)
            action, self.actionProbabilities = self.bee.policyFunc(
                actionValues, self.rng
            )
            self.state = self.patch.nextState(self.state, action)
            self.trajectory.append(self.state)
            self.done = self.state[:2] == self.patch.maximumIndex

        rows = [state[0] for state in self.trajectory]
        columns = [state[1] for state in self.trajectory]
        trajectoryX = self.patch.X[rows, columns]
        trajectoryY = self.patch.Y[rows, columns]

        self.axp.clear()
        self.axp.contourf(
            self.patch.X, self.patch.Y, self.patch.F,
            levels=np.linspace(0, 1, 11), cmap='coolwarm_r',
            vmin=0, vmax=1
        )
        self.axp.plot(trajectoryX, trajectoryY, color='black', linewidth=1.5)
        self.axp.scatter(
            trajectoryX[-1], trajectoryY[-1], s=45,
            facecolor='white', edgecolor='black', zorder=3
        )
        maximumRow, maximumColumn = self.patch.maximumIndex
        self.axp.scatter(
            self.patch.X[maximumRow, maximumColumn],
            self.patch.Y[maximumRow, maximumColumn],
            marker='*', s=140, facecolor='white', edgecolor='black'
        )
        status = 'maximum reached' if self.done else 'travelling'
        self.axp.set_title(
            f'Nectar={self.patch.nectarAt(self.state):.2f}; {status}'
        )
        self.axp.set_xlabel('x position')
        self.axp.set_ylabel('y position')

        self.axv.clear()
        self.axv.bar(
            np.arange(len(turnSteps)), self.actionProbabilities,
            color='0.45'
        )
        self.axv.set_xticks(np.arange(len(turnSteps)), turnLabels)
        self.axv.set_ylim(0, 1)
        self.axv.set_title('Softmax action policy')
        self.axv.set_xlabel('Relative turn (degrees)')
        self.axv.set_ylabel('p(action)')
        sns.despine(ax=self.axv)

    def animate(self):
        return FuncAnimation(
            self.fig, self.update, frames=self.simPars['tMax'],
            interval=200
        )

Let's simulate the bee for 80 time steps.

In [ ]:
mdpBee = BeeMDP(params={'softmaxInverseTemp': 20})
mdpSimPars = {
    'tMax': 80,
    'stateInit': (10, 10, 2),
    'seed': 2
}
mdpSimulator = SimulateBeeMDP(mdpBee, mdpPatch, mdpSimPars)
mdpAnimation = mdpSimulator.animate()
mdpHtml = mdpAnimation.to_jshtml()
plt.close(mdpSimulator.fig)
display(HTML(mdpHtml))

What do you notice?

> Under this policy, the bee sometimes reaches the global maximum and sometimes does not, depending on the random seed and starting position.
>
> Increasing the inverse temperature makes the route more deterministic. Decreasing it produces more exploration.

# 5. Selecting actions in a partially observable Markov decision process
Now let's return to the two flower species. Each location has a baseline mixture $q(x,y)=P(\mathrm{species\ A}\mid x,y)$, and species A contains more nectar on average than species B.  The bee observes its position and heading, but it does not directly observe which species generated the encountered flower.  This makes the state "partially observable", since we can observe the colour and the nectar rewards from those colours, which tell us something about the species, but we cannot observe the species itself.

In the one-species example, there was a positive linear relationship between flower colour and nectar.  However, in the second species, the relationship between colour and nectar is swapped!  More blue means higher nectar for species A, but lower nectar for species B.  So it is very important for the bee to know which species it is observing!  And it must infer this species just based on flower colour and how much nectar it gets from the flowers it samples!

The hidden state now includes species identity. Flower colour and nectar are noisy observations of that state, so the bee maintains a belief $b_t=P(c_t\mid f_{1:t},r_{1:t},a_{1:t-1},x_{1:t},y_{1:t})$. A POMDP can be controlled using this belief in place of the unavailable hidden state.

<div style="text-align:center">
    <img src="./model8.png" alt="Partially observable Markov decision process" width="900"/>
</div>

After choosing an action, the bee first predicts the species at the next location:

$$b^-_{t+1}(c')=\sum_c P(c'\mid c,a_t,x_{t+1},y_{t+1})b_t(c)$$

It then updates this prediction using the observed flower colour and nectar:

$$b_{t+1}(c')\propto p(f_{t+1}\mid c')p(r_{t+1}\mid f_{t+1},c')b^-_{t+1}(c')$$

In [ ]:
class POMDPFlowerPatch:
    def __init__(self, contexts, gridSize=21,
                 sigmaSensory=0.08, sigmaReward=0.15):
        self.contexts = contexts
        self.contextLabels = list(contexts)
        self.sigmaSensory = sigmaSensory
        self.sigmaReward = sigmaReward
        self.contextPersistence = np.array([0.2, 0.5, 0.9, 0.5, 0.2])

        coordinates = np.linspace(-3, 3, gridSize)
        self.X, self.Y = np.meshgrid(coordinates, coordinates)
        # Here the shared spatial field is the baseline P(species A).
        self.speciesAProbability = makeSpatialField(self.X, self.Y)
        self.richestIndex = np.unravel_index(
            np.argmax(self.speciesAProbability),
            self.speciesAProbability.shape
        )

        self.flowerMu = np.array([
            contexts[label].pFlowerMu for label in self.contextLabels
        ])
        self.observationSigma = np.array([
            np.sqrt(contexts[label].pFlowerSig**2 + sigmaSensory**2)
            for label in self.contextLabels
        ])

        fGrid = np.linspace(-0.2, 1.2, 500)
        self.expectedNectar = np.array([
            np.sum(
                self.nectar(fGrid, label)
                * norm.pdf(
                    fGrid,
                    loc=self.flowerMu[index],
                    scale=self.observationSigma[index]
                )
            )
            / np.sum(norm.pdf(
                fGrid,
                loc=self.flowerMu[index],
                scale=self.observationSigma[index]
            ))
            for index, label in enumerate(self.contextLabels)
        ])

    def nextState(self, state, action):
        row, column, heading = state
        headingNext = int(
            (heading + turnSteps[action]) % len(headingVectors)
        )
        rowStep, columnStep = headingVectors[headingNext]
        rowNext = int(
            np.clip(row + rowStep, 0, self.speciesAProbability.shape[0] - 1)
        )
        columnNext = int(
            np.clip(
                column + columnStep,
                0,
                self.speciesAProbability.shape[1] - 1
            )
        )
        return rowNext, columnNext, headingNext

    def localPrior(self, state):
        pSpeciesA = float(self.speciesAProbability[state[0], state[1]])
        return np.array([pSpeciesA, 1 - pSpeciesA])

    def nectar(self, flowerColor, context):
        if context == 'species_A':
            return norm.cdf(flowerColor, loc=0.5, scale=0.15)
        if context == 'species_B':
            return 0.5 * (
                1 - norm.cdf(flowerColor, loc=0.5, scale=0.15)
            )
        raise ValueError(f'unknown context: {context}')

    def transitionContext(self, contextIndex, stateNext, action, rng):
        transitionProbability = (
            (1 - self.contextPersistence[action])
            * self.localPrior(stateNext)
        )
        transitionProbability[contextIndex] += self.contextPersistence[action]
        return int(rng.choice(2, p=transitionProbability))

    def sampleObservation(self, contextIndex, rng):
        flowerColor = rng.normal(
            self.flowerMu[contextIndex],
            self.observationSigma[contextIndex]
        )
        context = self.contextLabels[contextIndex]
        reward = rng.normal(
            self.nectar(flowerColor, context), self.sigmaReward
        )
        return flowerColor, reward

Run and plot the results

In [ ]:
pomdpPatch = POMDPFlowerPatch(contexts, gridSize=21)
speciesColors = {'species_A': 'tab:blue', 'species_B': 'tab:red'}
extent = [
    pomdpPatch.X.min(), pomdpPatch.X.max(),
    pomdpPatch.Y.min(), pomdpPatch.Y.max()
]

fig, axs = plt.subplots(1, 3, figsize=(16, 4), layout='constrained')
mixtureImage = axs[0].imshow(
    pomdpPatch.speciesAProbability,
    origin='lower',
    extent=extent,
    interpolation='bilinear',
    cmap='coolwarm_r',
    vmin=0,
    vmax=1
)
fig.colorbar(
    mixtureImage, ax=axs[0],
    label=r'$P(A\mid x,y)$'
)
axs[0].set_title('Baseline species mixture')
axs[0].set_xlabel('x position')
axs[0].set_ylabel('y position')

fPlot = np.linspace(0, 1, 200)
for index, context in enumerate(pomdpPatch.contextLabels):
    axs[1].plot(
        fPlot,
        norm.pdf(
            fPlot,
            loc=pomdpPatch.flowerMu[index],
            scale=pomdpPatch.observationSigma[index]
        ),
        color=speciesColors[context],
        label=context
    )
axs[1].set_title('Flower-colour likelihood')
axs[1].set_xlabel('Observed flower colour')
axs[1].set_ylabel('Probability density')
axs[1].legend(frameon=False)

for context in pomdpPatch.contextLabels:
    axs[2].plot(
        fPlot,
        pomdpPatch.nectar(fPlot, context),
        color=speciesColors[context],
        label=context
    )
axs[2].set_title('Nectar function')
axs[2].set_xlabel('Observed flower colour')
axs[2].set_ylabel('Expected nectar')
axs[2].legend(frameon=False)
sns.despine(ax=axs[1])
sns.despine(ax=axs[2])
plt.show()

for context, expectedNectar in zip(
    pomdpPatch.contextLabels, pomdpPatch.expectedNectar
):
    print(f'{context}: mean nectar = {expectedNectar:.2f}')

We will keep the same one-step softmax policy used in the MDP. The difference is that the bee can no longer evaluate an action using a known species. Instead, it predicts its belief after each candidate action and computes expected nectar by averaging over species.

Continuing straight makes the local species more persistent, whereas a large turn samples more strongly from the species mixture at the destination. This is why the candidate action affects both position and the predicted species belief.

In [ ]:
class BeePOMDP(BeeMDP):
    def __init__(self, params, patch):
        super().__init__(params)
        self.patch = patch

    def predictBelief(self, belief, stateNext, action):
        persistence = self.patch.contextPersistence[action]
        return (
            persistence * belief
            + (1 - persistence) * self.patch.localPrior(stateNext)
        )

    def updateBelief(self, predictedBelief, flowerColor, reward):
        rewardMean = np.array([
            self.patch.nectar(flowerColor, context)
            for context in self.patch.contextLabels
        ])
        logPosterior = (
            np.log(np.clip(predictedBelief, 1e-300, None))
            + norm.logpdf(
                flowerColor,
                loc=self.patch.flowerMu,
                scale=self.patch.observationSigma
            )
            + norm.logpdf(
                reward,
                loc=rewardMean,
                scale=self.patch.sigmaReward
            )
        )
        posterior = np.exp(logPosterior - np.max(logPosterior))
        return posterior / posterior.sum()

    def actionValues(self, state, belief):
        values = []
        for action in range(len(turnSteps)):
            stateNext = self.patch.nextState(state, action)
            predictedBelief = self.predictBelief(
                belief, stateNext, action
            )
            values.append(predictedBelief @ self.patch.expectedNectar)
        return np.array(values)


class SimulateBeePOMDP:
    def __init__(self, bee, patch, simPars):
        self.bee = bee
        self.patch = patch
        self.simPars = simPars
        self.rng = np.random.default_rng(simPars['seed'])
        self.state = simPars['stateInit']

        self.belief = patch.localPrior(self.state)
        self.contextIndex = int(self.rng.choice(2, p=self.belief))
        self.flowerColor, self.reward = patch.sampleObservation(
            self.contextIndex, self.rng
        )
        self.belief = bee.updateBelief(
            self.belief, self.flowerColor, self.reward
        )

        self.trajectory = [self.state]
        self.contextHistory = [self.contextIndex]
        self.beliefHistory = [self.belief[0]]
        self.done = self.state[:2] == patch.richestIndex
        self.fig, (self.axp, self.axb) = plt.subplots(
            1, 2, figsize=(12, 5), layout='constrained'
        )

    def update(self, t):
        if not self.done:
            actionValues = self.bee.actionValues(self.state, self.belief)
            action, _ = self.bee.policyFunc(actionValues, self.rng)
            stateNext = self.patch.nextState(self.state, action)
            predictedBelief = self.bee.predictBelief(
                self.belief, stateNext, action
            )

            self.contextIndex = self.patch.transitionContext(
                self.contextIndex, stateNext, action, self.rng
            )
            self.flowerColor, self.reward = self.patch.sampleObservation(
                self.contextIndex, self.rng
            )
            self.belief = self.bee.updateBelief(
                predictedBelief, self.flowerColor, self.reward
            )
            self.state = stateNext
            self.trajectory.append(self.state)
            self.contextHistory.append(self.contextIndex)
            self.beliefHistory.append(self.belief[0])
            self.done = self.state[:2] == self.patch.richestIndex

        rows = [state[0] for state in self.trajectory]
        columns = [state[1] for state in self.trajectory]
        trajectoryX = self.patch.X[rows, columns]
        trajectoryY = self.patch.Y[rows, columns]

        self.axp.clear()
        self.axp.contourf(
            self.patch.X,
            self.patch.Y,
            self.patch.speciesAProbability,
            levels=np.linspace(0, 1, 11),
            cmap='coolwarm_r',
            vmin=0,
            vmax=1
        )
        self.axp.plot(trajectoryX, trajectoryY, color='black', linewidth=1.5)
        self.axp.scatter(
            trajectoryX[-1], trajectoryY[-1], s=45,
            facecolor='white', edgecolor='black', zorder=3
        )
        richestRow, richestColumn = self.patch.richestIndex
        self.axp.scatter(
            self.patch.X[richestRow, richestColumn],
            self.patch.Y[richestRow, richestColumn],
            marker='*', s=140, facecolor='white', edgecolor='black'
        )
        localProbability = self.patch.localPrior(self.state)[0]
        self.axp.set_title(
            rf'Baseline $P(A)={localProbability:.2f}$; '
            rf'belief $P(A)={self.belief[0]:.2f}$'
        )
        self.axp.set_xlabel('x position')
        self.axp.set_ylabel('y position')

        self.axb.clear()
        time = np.arange(len(self.beliefHistory))
        trueSpeciesA = np.array(self.contextHistory) == 0
        self.axb.step(
            time, trueSpeciesA.astype(float), where='post',
            color='black', linewidth=1.25, label='True species A'
        )
        self.axb.plot(
            time, self.beliefHistory,
            color=speciesColors['species_A'],
            linewidth=2, label=r'Belief $P(\mathrm{species\ A})$'
        )
        self.axb.set_ylim(-0.05, 1.05)
        self.axb.set_yticks([0, 1], ['species B', 'species A'])
        self.axb.set_title('Hidden species and belief')
        self.axb.set_xlabel('Encounter')
        self.axb.legend(frameon=False, loc='lower right')
        sns.despine(ax=self.axb)

    def animate(self):
        return FuncAnimation(
            self.fig, self.update, frames=self.simPars['tMax'],
            interval=200
        )

Now simulate and plot!

In [ ]:
pomdpBee = BeePOMDP(
    params={'softmaxInverseTemp': 20},
    patch=pomdpPatch
)
pomdpSimPars = {
    'tMax': 80,
    'stateInit': (10, 10, 2),
    'seed': 0
}
pomdpSimulator = SimulateBeePOMDP(
    pomdpBee, pomdpPatch, pomdpSimPars
)
pomdpAnimation = pomdpSimulator.animate()
pomdpHtml = pomdpAnimation.to_jshtml()
plt.close(pomdpSimulator.fig)
display(HTML(pomdpHtml))

What do you notice?

> The true species can change even when neighbouring flower colours overlap. The posterior belief varies continuously rather than switching at a fixed colour threshold, although strong evidence can still move it sharply.
>
> When the bee is confident it is in the richer species A, continuing through that region has high expected value. When evidence favours species B, reorientation can become more attractive.
>
> The MDP bee selected actions from observed state values. The POMDP bee selects the same kinds of actions from values averaged over its belief.

To learn more about these ideas, you can't go wrong with Sutton and Barto's [*Reinforcement Learning: An Introduction*](https://incompleteideas.net/book/bookdraft2016sep.pdf).